In [ ]:
# 00 - Importação e leitura do dataset oficial (abas PEDE2022/2023/2024)

import pandas as pd
import numpy as np

PATH_OFFICIAL = "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# lê as 3 abas oficiais
df22 = pd.read_excel(PATH_OFFICIAL, sheet_name="PEDE2022")
df23 = pd.read_excel(PATH_OFFICIAL, sheet_name="PEDE2023")
df24 = pd.read_excel(PATH_OFFICIAL, sheet_name="PEDE2024")

print("Shapes:", df22.shape, df23.shape, df24.shape)
display(df22.head(3))


Shapes: (860, 42) (1014, 48) (1156, 50)


,RA,Fase,Turma,Nome,Ano nasc,Idade 22,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,...,Inglês,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defas,Destaque IEG,Destaque IDA,Destaque IPV
0,RA-1,7,A,Aluno-1,2003,19,Menina,2016,Escola Pública,Ametista,...,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,7,A,Aluno-2,2005,17,Menina,2017,Rede Decisão,Ametista,...,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,7,A,Aluno-3,2005,17,Menina,2016,Rede Decisão,Ametista,...,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...


In [ ]:
# 01 - Normalização de colunas

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("__", "_", regex=False)
    )
    return df

df22 = normalize_cols(df22)
df23 = normalize_cols(df23)
df24 = normalize_cols(df24)

print("OK: colunas normalizadas")


OK: colunas normalizadas


In [ ]:
# 02 - Build do painel por ano (aluno-ano) - versão alinhada ao seu diagnóstico

import re
import pandas as pd

def pick_best_by_priority(df: pd.DataFrame, pref: str, year: int):
    """
    Escolhe a melhor coluna para o indicador 'pref' dado o ano.
    Regras:
      1) tenta achar coluna do ano (ex: INDE_2024, INDE_2023, INDE_22)
      2) tenta achar coluna genérica (ex: INDE, PEDRA, IDADE, IPP)
      3) fallback: qualquer coluna que contenha o prefixo
    Critério final: menor % de nulos (mais preenchida).
    """
    cols = list(df.columns)

    def norm(s):
        return re.sub(r"[^A-Z0-9_]", "_", str(s).upper())

    pref_n = norm(pref)
    year_str = str(year)

    # padrões do seu dataset: 2022 usa _22; 2023/2024 usam _2023/_2024
    year_short = year_str[-2:]

    candidates = []

    # 1) colunas do ano (forte)
    for c in cols:
        cn = norm(c)
        if cn.startswith(pref_n) and (f"_{year_str}" in cn or f"_{year_short}" in cn):
            candidates.append(c)

    # 2) coluna genérica
    for c in cols:
        if norm(c) == pref_n and c not in candidates:
            candidates.append(c)

    # 3) qualquer uma contendo o prefixo
    for c in cols:
        if pref_n in norm(c) and c not in candidates:
            candidates.append(c)

    if not candidates:
        return None

    # escolhe menor % de nulos
    best = min(candidates, key=lambda c: df[c].isna().mean())
    return best


def build_year_panel(df: pd.DataFrame, year: int) -> pd.DataFrame:
    df = df.copy()

    # colunas base (pega se existirem)
    base_cols = []
    for c in ["RA", "NOME", "FASE", "TURMA", "GENERO", "GÊNERO",
              "INSTITUICAO_DE_ENSINO", "INSTITUIÇÃO_DE_ENSINO",
              "INSTITUICAO_DE_ENSINO_ALUNO", "INSTITUIÇÃO_DE_ENSINO_ALUNO",
              "ANO_INGRESSO"]:
        if c in df.columns:
            base_cols.append(c)

    # indicadores que queremos no painel final (padronizados)
    indicators = [
        "IAN","IDA","IEG","IAA","IPS","IPV",
        "INDE","IPP","PEDRA","DEFASAGEM","IDADE"
    ]

    keep = base_cols.copy()
    rename_map = {}

    for pref in indicators:
        chosen = pick_best_by_priority(df, pref, year)
        if chosen is not None:
            keep.append(chosen)
            rename_map[chosen] = pref

    # remove duplicatas
    keep = pd.Index(keep).drop_duplicates().tolist()

    out = df.loc[:, keep].copy()
    out = out.rename(columns=rename_map)

    # padronizações extras
    if "GÊNERO" in out.columns and "GENERO" not in out.columns:
        out = out.rename(columns={"GÊNERO": "GENERO"})

    if "INSTITUIÇÃO_DE_ENSINO" in out.columns and "INSTITUICAO_DE_ENSINO" not in out.columns:
        out = out.rename(columns={"INSTITUIÇÃO_DE_ENSINO": "INSTITUICAO_DE_ENSINO"})
    if "INSTITUICAO_DE_ENSINO_ALUNO" in out.columns and "INSTITUICAO_DE_ENSINO" not in out.columns:
        out = out.rename(columns={"INSTITUICAO_DE_ENSINO_ALUNO": "INSTITUICAO_DE_ENSINO"})

    out["ANO"] = int(year)
    out = out.loc[:, ~out.columns.duplicated()].copy()

    return out


In [ ]:
# 03 - Gerar painel aluno-ano (2022/2023/2024) e validar

p22 = build_year_panel(df22, 2022)
p23 = build_year_panel(df23, 2023)
p24 = build_year_panel(df24, 2024)


print("p22:", p22.shape, "| duplicadas:", p22.columns.duplicated().sum())
print("p23:", p23.shape, "| duplicadas:", p23.columns.duplicated().sum())
print("p24:", p24.shape, "| duplicadas:", p24.columns.duplicated().sum())

panel = pd.concat([p22, p23, p24], ignore_index=True)

print("panel:", panel.shape)
display(panel.head())

# checagens básicas
assert "RA" in panel.columns, "RA não apareceu no painel"
assert "ANO" in panel.columns, "ANO não apareceu no painel"
print("OK: painel montado")


p22: (860, 16) | duplicadas: 0
p23: (1014, 17) | duplicadas: 0
p24: (1156, 17) | duplicadas: 0
panel: (3030, 18)


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN
3,RA-4,Aluno-4,7,A,Menino,Rede Decisão,2017,10.0,17,4.5,8.8,5.6,5.278,5.951,Quartzo,2022,NaN,NaN
4,RA-5,Aluno-5,7,A,Menina,Rede Decisão,2016,10.0,17,8.6,7.9,5.6,7.389,7.427,Ametista,2022,NaN,NaN


OK: painel montado


In [ ]:
# 04 - Harmonizar colunas críticas por ano (caso alguma venha com nome diferente)

def _strip_accents_upper(s: str) -> str:
    # versão simples sem libs externas
    repl = {
        "Ç":"C","Ã":"A","Á":"A","Â":"A","À":"A",
        "É":"E","Ê":"E","È":"E",
        "Í":"I","Ì":"I",
        "Ó":"O","Ô":"O","Õ":"O","Ò":"O",
        "Ú":"U","Ù":"U"
    }
    out = s
    for k,v in repl.items():
        out = out.replace(k, v)
    return out

def _variants(col: str):
    col = col.upper()
    v = set([col])
    v.add(_strip_accents_upper(col))
    v.add(col.replace("INSTITUIÇÃO", "INSTITUICAO"))
    v.add(_strip_accents_upper(col.replace("INSTITUIÇÃO", "INSTITUICAO")))
    return list(v)

YEAR_RULES = {
    2022: {
        "INDE":      _variants("INDE_2022") + _variants("INDE"),
        "IPP":       _variants("IPP_2022") + _variants("IPP"),
        "PEDRA":     _variants("PEDRA_2022") + _variants("PEDRA"),
        "DEFASAGEM": _variants("DEFASAGEM_2022") + _variants("DEFASAGEM"),
        "IDADE":     _variants("IDADE_ALUNO_2022") + _variants("IDADE_2022") + _variants("IDADE"),
        "INSTITUICAO_DE_ENSINO": (
            _variants("INSTITUICAO_ENSINO_ALUNO_2022")
            + _variants("INSTITUICAO_DE_ENSINO_ALUNO_2022")
            + _variants("INSTITUICAO_DE_ENSINO")
        ),
    },
    2023: {
        "INDE":      _variants("INDE_2023") + _variants("INDE"),
        "IPP":       _variants("IPP_2023") + _variants("IPP"),
        "PEDRA":     _variants("PEDRA_2023") + _variants("PEDRA"),
        "DEFASAGEM": _variants("DEFASAGEM_2023") + _variants("DEFASAGEM"),
        "IDADE":     _variants("IDADE_ALUNO_2023") + _variants("IDADE_2023") + _variants("IDADE"),
        "INSTITUICAO_DE_ENSINO": (
            _variants("INSTITUICAO_ENSINO_ALUNO_2023")
            + _variants("INSTITUICAO_DE_ENSINO_ALUNO_2023")
            + _variants("INSTITUICAO_DE_ENSINO")
        ),
    },
    2024: {
        "INDE":      _variants("INDE_2024") + _variants("INDE"),
        "IPP":       _variants("IPP_2024") + _variants("IPP"),
        "PEDRA":     _variants("PEDRA_2024") + _variants("PEDRA"),
        "DEFASAGEM": _variants("DEFASAGEM_2024") + _variants("DEFASAGEM"),
        "IDADE":     _variants("IDADE_ALUNO_2024") + _variants("IDADE_2024") + _variants("IDADE"),
        "INSTITUICAO_DE_ENSINO": (
            _variants("INSTITUICAO_ENSINO_ALUNO_2024")
            + _variants("INSTITUICAO_DE_ENSINO_ALUNO_2024")
            + _variants("INSTITUICAO_DE_ENSINO")
        ),
    },
}

def harmonize_year_specific_cols(df_year: pd.DataFrame, year: int) -> pd.DataFrame:
    df_year = df_year.copy()
    rules = YEAR_RULES.get(int(year), {})

    for target, candidates in rules.items():
        if target in df_year.columns:
            continue

        found = None
        for c in candidates:
            if c in df_year.columns:
                found = c
                break

        if found is not None:
            df_year = df_year.rename(columns={found: target})

    df_year = df_year.loc[:, ~df_year.columns.duplicated()].copy()
    return df_year

panel_fixed = []
for y, part in panel.groupby("ANO", sort=True):
    panel_fixed.append(harmonize_year_specific_cols(part, int(y)))

panel = pd.concat(panel_fixed, ignore_index=True)

# checagem de preenchimento
critical = ["INDE","PEDRA","IPP","DEFASAGEM","IDADE","INSTITUICAO_DE_ENSINO"]
for y in sorted(panel["ANO"].unique()):
    tmp = panel[panel["ANO"] == y]
    stats = {c: (tmp[c].isna().mean() if c in tmp.columns else None) for c in critical}
    print(f"ANO {y} | % nulos:", {k: (None if v is None else round(v,3)) for k,v in stats.items()})

display(panel.head(3))


ANO 2022 | % nulos: {'INDE': np.float64(0.0), 'PEDRA': np.float64(0.0), 'IPP': np.float64(1.0), 'DEFASAGEM': np.float64(1.0), 'IDADE': np.float64(0.0), 'INSTITUICAO_DE_ENSINO': np.float64(0.0)}
ANO 2023 | % nulos: {'INDE': np.float64(0.082), 'PEDRA': np.float64(0.082), 'IPP': np.float64(0.075), 'DEFASAGEM': np.float64(0.0), 'IDADE': np.float64(0.0), 'INSTITUICAO_DE_ENSINO': np.float64(0.0)}
ANO 2024 | % nulos: {'INDE': np.float64(0.055), 'PEDRA': np.float64(0.055), 'IPP': np.float64(0.088), 'DEFASAGEM': np.float64(0.0), 'IDADE': np.float64(0.0), 'INSTITUICAO_DE_ENSINO': np.float64(0.001)}


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN


In [ ]:
# 05 - Salvar datamarts (wide + long)

# Wide (aluno-ano)
panel = panel.copy()
panel["ANO"] = panel["ANO"].astype(int)
panel["RA"] = panel["RA"].astype(str).str.strip()

panel.to_csv("datamart_wide_aluno_ano.csv", index=False)
print("✅ Salvo: datamart_wide_aluno_ano.csv", panel.shape)

# Long (indicadores empilhados)
id_cols = [c for c in ["RA","NOME","ANO","FASE","TURMA","GENERO","INSTITUICAO_DE_ENSINO","ANO_INGRESSO","IDADE","PEDRA"] if c in panel.columns]
value_cols = [c for c in ["INDE","IAN","IDA","IEG","IAA","IPS","IPP","IPV"] if c in panel.columns]

long_df = panel.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="INDICADOR",
    value_name="VALOR"
)

long_df.to_csv("datamart_long_indicadores.csv", index=False)
print("✅ Salvo: datamart_long_indicadores.csv", long_df.shape)

display(long_df.head(5))


✅ Salvo: datamart_wide_aluno_ano.csv (3030, 18)
✅ Salvo: datamart_long_indicadores.csv (21210, 12)


,RA,NOME,ANO,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IDADE,PEDRA,INDICADOR,VALOR
0,RA-1,Aluno-1,2022,7,A,Menina,Escola Pública,2016,19,Quartzo,INDE,5.783
1,RA-2,Aluno-2,2022,7,A,Menina,Rede Decisão,2017,17,Ametista,INDE,7.055
2,RA-3,Aluno-3,2022,7,A,Menina,Rede Decisão,2016,17,Ágata,INDE,6.591
3,RA-4,Aluno-4,2022,7,A,Menino,Rede Decisão,2017,17,Quartzo,INDE,5.951
4,RA-5,Aluno-5,2022,7,A,Menina,Rede Decisão,2016,17,Ametista,INDE,7.427


In [ ]:
# Diagnóstico: quais colunas existem e qual % de nulos em cada ano (antes do build)

def show_candidates(df, year, prefixes=("INDE","IPP","PEDRA","DEFASAGEM","IDADE")):
    print(f"\n==== PEDE{year} - candidatos ====")
    for pref in prefixes:
        # pega qualquer coluna que contenha o prefixo
        cands = [c for c in df.columns if pref in c]
        print(f"\n{pref}: {cands if cands else 'NENHUMA'}")
        for c in cands[:15]:
            null_rate = df[c].isna().mean()
            print(f"  - {c}: %nulos={null_rate:.3f}")

show_candidates(df22, 2022)
show_candidates(df23, 2023)
show_candidates(df24, 2024)



==== PEDE2022 - candidatos ====

INDE: ['INDE_22']
  - INDE_22: %nulos=0.000

IPP: NENHUMA

PEDRA: ['PEDRA_20', 'PEDRA_21', 'PEDRA_22']
  - PEDRA_20: %nulos=0.624
  - PEDRA_21: %nulos=0.463
  - PEDRA_22: %nulos=0.000

DEFASAGEM: NENHUMA

IDADE: ['IDADE_22']
  - IDADE_22: %nulos=0.000

==== PEDE2023 - candidatos ====

INDE: ['INDE_2023', 'INDE_22', 'INDE_23']
  - INDE_2023: %nulos=0.082
  - INDE_22: %nulos=0.408
  - INDE_23: %nulos=1.000

IPP: ['IPP']
  - IPP: %nulos=0.075

PEDRA: ['PEDRA_2023', 'PEDRA_20', 'PEDRA_21', 'PEDRA_22', 'PEDRA_23']
  - PEDRA_2023: %nulos=0.082
  - PEDRA_20: %nulos=0.763
  - PEDRA_21: %nulos=0.670
  - PEDRA_22: %nulos=0.408
  - PEDRA_23: %nulos=1.000

DEFASAGEM: ['DEFASAGEM']
  - DEFASAGEM: %nulos=0.000

IDADE: ['IDADE']
  - IDADE: %nulos=0.000

==== PEDE2024 - candidatos ====

INDE: ['INDE_2024', 'INDE_22', 'INDE_23']
  - INDE_2024: %nulos=0.055
  - INDE_22: %nulos=0.592
  - INDE_23: %nulos=0.403

IPP: ['IPP']
  - IPP: %nulos=0.088

PEDRA: ['PEDRA_2024', 'PE

In [ ]:
for y in [2022, 2023, 2024]:
    tmp = panel[panel["ANO"] == y]
    stats = {c: tmp[c].isna().mean() for c in ["INDE","PEDRA","IDADE","IPP","DEFASAGEM"] if c in tmp.columns}
    print(f"ANO {y} | % nulos:", {k: round(v,3) for k,v in stats.items()})


ANO 2022 | % nulos: {'INDE': np.float64(0.0), 'PEDRA': np.float64(0.0), 'IDADE': np.float64(0.0), 'IPP': np.float64(1.0), 'DEFASAGEM': np.float64(1.0)}
ANO 2023 | % nulos: {'INDE': np.float64(0.082), 'PEDRA': np.float64(0.082), 'IDADE': np.float64(0.0), 'IPP': np.float64(0.075), 'DEFASAGEM': np.float64(0.0)}
ANO 2024 | % nulos: {'INDE': np.float64(0.055), 'PEDRA': np.float64(0.055), 'IDADE': np.float64(0.0), 'IPP': np.float64(0.088), 'DEFASAGEM': np.float64(0.0)}
